Hourly mann-whitney analysis and supporting box plots

In [1]:
# For handling the timeseries
import pandas as pd, os, datetime
import numpy as np

# Statistical analysis
from scipy import stats
from scipy.stats import mannwhitneyu

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import matplotlib.lines as mlines
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import seaborn as sns
pio.renderers.default = 'notebook'

# Import the loader function from your module
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

In [ ]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    wind_zero_threshold=40,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

In [ ]:
def hourly_mannwhitu(df, var):
    df = df.copy()
    results = []
    df['hour'] = df['time'].dt.hour
    
    for duid, group in df.groupby('DUID'):
        for hour in range(24):
            group_hour = group[group['hour'] == hour]
            
            group_0 = group_hour[group_hour['EHF_flag'] == 0][var].dropna()
            group_1 = group_hour[group_hour['EHF_flag'] == 1][var].dropna()
            
            # print(f"DUID: {duid}, hour: {hour}, size 0: {len(group_0)}, size 1: {len(group_1)}")
            
            if len(group_0) < 10 or len(group_1) < 10:
                continue
            
            t_stat, p_val = mannwhitneyu(group_0, group_1, alternative='two-sided')
            
            results.append({
                'DUID': duid,
                'hour': hour,
                't_statistic': t_stat,
                'p_value': p_val,
                'median_baseline': group_0.median(),
                'median_hw': group_1.median()
            })
    
    result_df = pd.DataFrame(results)
    return result_df

In [ ]:
def plot_hourly_mannwhitu_map(df, info, mannwhitu_results, alpha=0.05,ftypes='Wind and Solar'):
    merged = mannwhitu_results.merge(info, on='DUID', how='left').dropna(subset=['lat_jittered', 'lon_jittered'])
    merged['change'] = merged['median_hw'] - merged['median_baseline']
    merged['size'] = (merged['change'].abs().replace(0, 0.001) * 100) + 20

    def pvalue_significance(p):
        if p < alpha / 10:
            return 'Very strong'
        elif p < alpha:
            return 'Strong'
        elif p < 0.1:
            return 'Weak'
        else:
            return 'Not significant'

    merged['SignificanceLevel'] = merged['p_value'].apply(pvalue_significance)

    def category(row):
        sig = row['SignificanceLevel']
        if sig == 'Not significant':
            return sig
        elif row['change'] < 0:
            return f'{sig} decrease'
        else:
            return f'{sig} increase'

    merged['Category'] = merged.apply(category, axis=1)

    color_map = {
        'Very strong increase': 'darkgreen',
        'Strong increase': 'green',
        'Weak increase': 'lightgreen',
        'Very strong decrease': 'darkred',
        'Strong decrease': 'red',
        'Weak decrease': 'salmon',
        'Not significant': 'lightgray'
    }

    # Ensure 'hour' is string for animation frame to work nicely
    merged['hour'] = merged['hour'].astype(str)

    fig = px.scatter_map(
        merged,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Category',
        color_discrete_map=color_map,
        hover_name='DUID',
        hover_data={
            'hour': True,
            'change': ':.2f',
            'median_baseline': ':.2f',
            'median_hw': ':.2f',
            'p_value': ':.4f',
            'lat_jittered': False,
            'lon_jittered': False,
            'size': False
        },
        zoom=2,
        map_style='carto-darkmatter',
        animation_frame='hour',
        title=f'Hourly Results: Shift in generation distribution for {ftypes}'
    )

    fig.update_layout(
        legend_title_text='Significance and Direction',
        margin=dict(l=10, r=10, t=50, b=10),
        map=dict(center=dict(lat=merged['lat_jittered'].median(), lon=merged['lon_jittered'].median()))
    )

    fig.show()
    return fig


In [ ]:
mannwhitu_hourly_results = hourly_mannwhitu(df,'TOTALMWh')
fig = plot_hourly_mannwhitu_map(df, info, mannwhitu_hourly_results,ftypes='Wind')
fig.write_html("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/hourly_mannwhitu_wind_plotly.html")

In [ ]:
def plot_hourly_mannwhitu_map_matplotlib(df, info, mannwhitu_results, alpha=0.05, ftypes='Wind'):
    merged = mannwhitu_results.merge(info, on='DUID', how='left').dropna(subset=['lat_jittered', 'lon_jittered'])
    merged['change'] = merged['median_hw'] - merged['median_baseline']
    merged['size'] = (merged['change'].abs().replace(0, 1) * 2) + 20

    def pvalue_significance(p):
        if p < alpha / 10:
            return 'Very strong'
        elif p < alpha:
            return 'Strong'
        elif p < 0.1:
            return 'Weak'
        else:
            return 'Not significant'

    merged['SignificanceLevel'] = merged['p_value'].apply(pvalue_significance)

    def category(row):
        sig = row['SignificanceLevel']
        if sig == 'Not significant':
            return sig
        elif row['change'] < 0:
            return f'{sig} decrease'
        else:
            return f'{sig} increase'

    merged['Category'] = merged.apply(category, axis=1)

    color_map = {
        'Very strong increase': 'darkgreen',
        'Strong increase': 'green',
        'Weak increase': 'lightgreen',
        'Very strong decrease': 'darkred',
        'Strong decrease': 'red',
        'Weak decrease': 'salmon',
        'Not significant': 'lightgray'
    }

    # Prepare for faceting by hour
    hours = sorted(merged['hour'].unique())
    n_hours = len(hours)
    ncols = 6
    nrows = int(np.ceil(n_hours / ncols))

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(2*ncols, 3*nrows),
        subplot_kw={'projection': ccrs.PlateCarree()},
        constrained_layout=True  # Use constrained_layout for better spacing
    )

    axes = axes.flatten()

    for idx, hour in enumerate(hours):
        ax = axes[idx]
        data = merged[merged['hour'] == hour]
        for cat, color in color_map.items():
            subset = data[data['Category'] == cat]
            ax.scatter(
                subset['lon_jittered'],
                subset['lat_jittered'],
                s=subset['size'],
                color=color,
                label=cat,
                alpha=0.8,
                edgecolor='k',
                linewidth=0.5,
                transform=ccrs.PlateCarree()
            )
        ax.set_title(f'Hour {hour}', fontsize=10)
        ax.coastlines(lw=0.5)
        ax.add_feature(cfeature.STATES, linestyle='-', lw=0.5)
        ax.set_extent([
            merged['lon_jittered'].min() - 1, merged['lon_jittered'].max() + 1,
            merged['lat_jittered'].min() - 1, merged['lat_jittered'].max() + 1
        ])
        if idx == 0:
            handles, labels = ax.get_legend_handles_labels()

    # Remove empty subplots
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    # Create custom legend handles with fixed marker size
    legend_handles = []
    for cat, color in color_map.items():
        handle = mlines.Line2D(
            [], [], 
            color=color, 
            marker='o', 
            linestyle='None',
            markersize=10,   # Set desired legend marker size here
            markeredgecolor='k',
            label=cat
        )
        legend_handles.append(handle)
    
    # Add a single legend just outside the top right
    fig.legend(
        handles=legend_handles,
        title='Significance and Direction',
        loc='upper right',
        bbox_to_anchor=(1.25, 1.0),
        fontsize=12,           # Make legend labels bigger
        title_fontsize=14      # Make legend title bigger
    )
    fig.suptitle(
        f'Hourly Results: Shift in generation distribution for {ftypes} (AEST)',
        fontsize=16,
        y=1.025
    )

    plt.show()
    return fig

fig = plot_hourly_mannwhitu_map_matplotlib(df, info, mannwhitu_hourly_results)
fig.savefig("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/hourly_mannwhitu_wind.png")

In [ ]:
def save_violinplots_by_duid_ehf_hourly(df, fuel_type=['Solar','Wind'], output_dir="violinplots", overlay_swarm=True):
    """
    Generate and save violin plots for each DUID split by EHF_flag (0 vs 1), for each hour of day.
    Each hour's plots are saved in a separate subdirectory.
    """ 
    os.makedirs(output_dir, exist_ok=True)

    # Filter data by fuel type
    df_filtered = df[df['fuel_source_primary'].isin(fuel_type)].copy()

    # Ensure 'time' is datetime
    if not np.issubdtype(df_filtered['time'].dtype, np.datetime64):
        df_filtered['time'] = pd.to_datetime(df_filtered['time'])

    # Add hour column
    df_filtered['hour'] = df_filtered['time'].dt.hour

    for hour in sorted(df_filtered['hour'].unique()):
        hour_df = df_filtered[df_filtered['hour'] == hour]
        hour_dir = os.path.join(output_dir, f"hour_{hour:02d}")
        os.makedirs(hour_dir, exist_ok=True)

        for duid in hour_df['DUID'].unique():
            duid_df = hour_df[hour_df['DUID'] == duid].copy()
            if duid_df.empty:
                continue

            fig, ax = plt.subplots(figsize=(6, 6))
            label_map = {0: 'Baseline', 1: 'Heatwave'}
            duid_df['EHF_flag_label'] = duid_df['EHF_flag'].map(label_map)

            sns.violinplot(
                x='EHF_flag_label',
                y='TOTALMWh',
                hue='EHF_flag_label',
                data=duid_df,
                palette={'Baseline': "skyblue", 'Heatwave': "orange"},
                inner='quartile',
                ax=ax,
                legend=False
            )

            if overlay_swarm:
                sns.stripplot(
                    x='EHF_flag_label',
                    y='TOTALMWh',
                    data=duid_df,
                    color='k',
                    alpha=0.5,
                    jitter=0.25,
                    size=2,
                    ax=ax
                )

            ax.set_title(f"Violin Plot for {duid} by Heatwave Status (Hour {hour:02d})")
            ax.set_ylabel("Energy generation during hour (MWh)")

            filepath = os.path.join(hour_dir, f"{duid}_violin_dist_{hour:02d}.png")
            plt.savefig(filepath, dpi=150, bbox_inches="tight")
            plt.close(fig)

    print(f"Saved violin plots by EHF_flag and hour for {len(df_filtered['DUID'].unique())} DUIDs to folder: {output_dir}")

# Example usage:
save_violinplots_by_duid_ehf_hourly(df, fuel_type=['Wind'], output_dir="/g/data/ng72/ms5578/ID_HW_BARRA/data/output/wind_chapter/violinplot")